# quantum-cortex — N2 Slice C: MQAR ablation (control / hopfield / delta)

Full difficulty curriculum on the three C2 variants. Produces the three accuracy curves and the verdict (which variant recalls further on MQAR). Results are a **separate artefact** (`metrics/mqar/`), not a run-v1 record. No Internet needed for MQAR (synthetic); GPU optional but faster.

Settings: **GPU T4** recommended, **Internet On** (to clone the private repo).

In [ ]:
from kaggle_secrets import UserSecretsClient
token = UserSecretsClient().get_secret("GITHUB_TOKEN")
!git clone https://{token}@github.com/vaneeckhoutnicolas/quantum-cortex.git
%cd quantum-cortex
!pip -q install numpy jsonschema
!pip -q install torch --index-url https://download.pytorch.org/whl/cpu  # or use the GPU torch already on Kaggle

In [ ]:
# Full MQAR ablation across the three variants. ~1500 steps/tier × 12 tiers × 3 variants.
# On GPU T4 this is a couple of hours; the circuit breaker aborts any diverging tier.
!python -m cortex_eval.ablation_mqar --steps 1500 --out metrics/mqar

In [ ]:
# Read back the verdict and the curves
import json, glob, os
latest = json.load(open('metrics/mqar/LATEST.json'))
print('VERDICT:', latest['verdict'])
print('drop-offs:', json.dumps(latest['dropoffs'], indent=2))
print('vs control:', json.dumps(latest['control_delta'], indent=2))
print()
# print the full curves per variant
for v, curve in latest['curves'].items():
    print(f'--- {v} ---')
    for pt in curve:
        print(f"  kv={pt['kv_pairs']:>2} seq={pt['seq_len']:>4}  acc={pt['accuracy']:.3f}")
print()
print('Artefact file (download from Output, then commit under metrics/mqar/):')
print(sorted(glob.glob('metrics/mqar/mqar-ablation-*.json'))[-1])